# Car-Census on Google Colab

Count cars in a traffic video and identify their make, model, and generation.
See the [README](https://github.com/DmitryMatv/Car-Census) for full details.

Before running: **Runtime > Change runtime type > T4 GPU**.

Without a polygon ROI, the full frame is used as the counting zone, so the
notebook works out of the box. The first run downloads the pretrained
RF-DETR-M checkpoint (~100 MB).

## 1. Clone and install

In [ ]:
!git clone https://github.com/DmitryMatv/Car-Census.git
%cd Car-Census
%pip install -q uv
# --system targets Colab's interpreter (no venv); fall back to %pip install -e . if uv misbehaves
!uv pip install --system -e .
!apt-get -qq install -y fonts-noto-color-emoji

## 2. Add a video

The repo ships no sample footage, so upload your own clip (30 fps works best).
To use a video from Google Drive instead, mount the drive and copy it into
`input_data/` yourself.

In [ ]:
import pathlib
import shutil

from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

VIDEO = next(iter(uploaded))
pathlib.Path("input_data").mkdir(exist_ok=True)
shutil.move(VIDEO, f"input_data/{VIDEO}")
print(f"Using video: input_data/{VIDEO}")

## 3. (Optional) TrafficEye API key

Make/model recognition calls the TrafficEye API. Add `TRAFFICEYE_API_KEY` in
the Colab Secrets panel (key icon on the left), or skip this cell and run with
`--skip-classify` below.

In [ ]:
import os

from google.colab import userdata

try:
    os.environ["TRAFFICEYE_API_KEY"] = userdata.get("TRAFFICEYE_API_KEY")
    print("TrafficEye key loaded. Classification will run.")
except Exception:
    print("No TRAFFICEYE_API_KEY secret found. Use --skip-classify below.")

## 4. Run the pipeline

Detects and tracks vehicles, counts them, and exports `report.csv`.
Add `--skip-render` if the annotated video is not needed; rendering is the
slowest stage on Colab.

In [ ]:
!Car-Census run input_data/{VIDEO} --accelerator colab-t4 --skip-classify

In [ ]:
# Full run with TrafficEye make/model recognition (needs the API key above):
!Car-Census run input_data/{VIDEO} --accelerator colab-t4

## 5. Get results

Each run writes to `output/<run-id>/`.

In [ ]:
import glob
import os
import shutil

from google.colab import files

run_dir = os.path.dirname(sorted(glob.glob("output/*/run.json"))[-1])
run_name = os.path.basename(run_dir)
print(f"Latest run: {run_dir}")
!ls {run_dir}

files.download(f"{run_dir}/report.csv")
# The annotated video can be large:
# files.download(f"{run_dir}/annotated.mp4")

# Whole run folder (crops, labels, artifacts) as one zip:
archive = shutil.make_archive(run_name, "zip", "output", run_name)
print(f"Archive: {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")
files.download(archive)

## Next steps

- Restrict counting to a polygon zone with a camera profile: run
  `Car-Census roi edit input_data/{VIDEO} --camera-id my-camera`, then rerun
  with `--camera-id my-camera` (interactive; needs a display, so do this
  locally).
- Overlay custom settings with `--config PATH`; see the README's
  Configuration section.
- Reuse API results across runs with the retrieval cache (`cache seed`,
  `cache calibrate`); see the README's TrafficEye Setup section.